# Alpha-beta pruning

_Artificial Intelligence (CS221)_

**Skip branches that cannot change your decision. Same answer, far less work.**

Minimax explores the whole game tree. That can be enormous.

     Alpha-beta pruning skips branches that cannot possibly affect the final choice.

---

This notebook is a practice scaffold for the **Alpha-beta pruning** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — Python

In [ ]:
tree = [[[3, 12], [8, 2]], [[4, 6], [14, 5]]]
visited = [0]                            # mutable counter of leaves examined

def ab(node, alpha, beta, maximizing):
    if isinstance(node, int):
        visited[0] += 1
        return node
    if maximizing:
        v = -10**9
        for child in node:
            v = max(v, ab(child, alpha, beta, False))
            alpha = max(alpha, v)
            if alpha >= beta:
                break                    # beta cutoff: prune the rest
        return v
    else:
        v = 10**9
        for child in node:
            v = min(v, ab(child, alpha, beta, True))
            beta = min(beta, v)
            if alpha >= beta:
                break                    # alpha cutoff: prune the rest
        return v

value = ab(tree, -10**9, 10**9, True)
print("alpha-beta value:", value)
print("leaves examined :", visited[0], "of 8 total (rest were pruned)")

## Visualize the data & results

_On that same tic-tac-toe position, how many game positions does alpha-beta skip versus full minimax?_

In [ ]:
import matplotlib.pyplot as plt

def winner(b):
    lines = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
    for a, c, d in lines:
        if b[a] != " " and b[a] == b[c] == b[d]: return b[a]
    return None
board = ["O","O"," "," ","X"," "," "," "," "]
counts = {"full": 0, "ab": 0}

def full(b, player):
    counts["full"] += 1
    if winner(b) or " " not in b: return
    for i in range(9):
        if b[i] == " ": full(b[:i]+[player]+b[i+1:], "O" if player=="X" else "X")
def ab(b, player, alpha, beta):
    counts["ab"] += 1
    w = winner(b)
    if w == "X": return 1
    if w == "O": return -1
    if " " not in b: return 0
    if player == "X":
        v = -2
        for i in range(9):
            if b[i] == " ":
                v = max(v, ab(b[:i]+["X"]+b[i+1:], "O", alpha, beta)); alpha = max(alpha, v)
                if alpha >= beta: break
        return v
    v = 2
    for i in range(9):
        if b[i] == " ":
            v = min(v, ab(b[:i]+["O"]+b[i+1:], "X", alpha, beta)); beta = min(beta, v)
            if alpha >= beta: break
    return v

full(board, "X"); ab(board, "X", -2, 2)
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["alpha-beta","full minimax"], [counts["ab"], counts["full"]],
       color=["#7ee787","#ffb454"])
ax.text(0, counts["ab"], str(counts["ab"]), ha="center", va="bottom")
ax.text(1, counts["full"], str(counts["full"]), ha="center", va="bottom")
ax.set_title("positions examined: alpha-beta vs minimax")
plt.show()